# 01 - Setup repo and download

In [ ]:
########### ----------- Exercise 01 ----------- ###########

'''
    I already use a repo so I will just continoue in this
'''

import requests


def download_file(url, output_path):
    response = requests.get(url)
    response.raise_for_status()
    with open(output_path, "wb") as f:
        f.write(response.content)

download_file("https://teaching.healthtech.dtu.dk/material/22118/scores.txt", "scores.txt")
download_file("https://teaching.healthtech.dtu.dk/material/22118/negative_list.txt", "negative_list.txt")
download_file("https://teaching.healthtech.dtu.dk/material/22118/translation.txt", "translation.txt")

# 02

The input file scores.txt is a tab-separated file with an accession number in first column followed by 6 numbers (scores) between 0 and 1. 
You must find the accession numbers and scores (that means the entire line) of the 10 highest and 10 lowest "combined scores" (combined score is the metric for selection) and save the output in the file scoresextreme.txt.
The combined score is simply the 6 numbers added together. The order of the output must be from high to low.

In [18]:
import pandas as pd

def main():
    filename = "scores.txt"

    # Load data
    df = pd.read_csv(filename, sep="\t", names=['id', '1', '2', '3', '4', '5', '6'])

    # Sum cols and order
    df["Total"] = df.iloc[:, 1:].sum(axis=1)
    df = df[['id', 'Total']]
    df = df.sort_values(by="Total", ascending=False)

    # Save the top 10 in a new df
    df_top = df.head(10)
    df_tail = df.tail(10)
    df_extreme = pd.concat([df_top, df_tail])
    df_extreme.to_csv("scores_extreme.txt", sep="\t", index=False)

    print(df_extreme)

main()


'''
id	Total
LF808489.CDS.1	5.27661
KI133588.CDS.2	5.24153
JB80314549.CDS.3	5.178749999999999
J21782.CDS.3	5.1575500000000005
UY37981324.CDS.1	5.11245
CR97613151.CDS.1	5.09985
TF58157495	5.07904
N33521.CDS.1	5.037459999999999
E15922.CDS.2	5.02731
HD25632497.CDS.1	5.004689999999999
JP12357726.CDS.1	1.1492099999999998
IE553374.CDS.1	1.1350500000000001
FS53962190.CDS.1	1.12183
ZL767849.CDS.1	1.07654
QS900083.CDS.1	1.05693
ND80117744.CDS.1	1.04862
KP22241601.CDS.2	0.9468399999999999
YM29518703.CDS.1	0.9122
BQ99602892.CDS.1	0.7989900000000001
BE618688.CDS.1	0.78614
'''


                    id    Total
3643    LF808489.CDS.1  5.27661
0       KI133588.CDS.2  5.24153
2310  JB80314549.CDS.3  5.17875
4836      J21782.CDS.3  5.15755
416   UY37981324.CDS.1  5.11245
432   CR97613151.CDS.1  5.09985
3868        TF58157495  5.07904
2795      N33521.CDS.1  5.03746
2658      E15922.CDS.2  5.02731
2842  HD25632497.CDS.1  5.00469
3902  JP12357726.CDS.1  1.14921
2769    IE553374.CDS.1  1.13505
4796  FS53962190.CDS.1  1.12183
3313    ZL767849.CDS.1  1.07654
1136    QS900083.CDS.1  1.05693
2666  ND80117744.CDS.1  1.04862
3937  KP22241601.CDS.2  0.94684
1683  YM29518703.CDS.1  0.91220
3572  BQ99602892.CDS.1  0.79899
1601    BE618688.CDS.1  0.78614


'\nid\tTotal\nLF808489.CDS.1\t5.27661\nKI133588.CDS.2\t5.24153\nJB80314549.CDS.3\t5.178749999999999\nJ21782.CDS.3\t5.1575500000000005\nUY37981324.CDS.1\t5.11245\nCR97613151.CDS.1\t5.09985\nTF58157495\t5.07904\nN33521.CDS.1\t5.037459999999999\nE15922.CDS.2\t5.02731\nHD25632497.CDS.1\t5.004689999999999\nJP12357726.CDS.1\t1.1492099999999998\nIE553374.CDS.1\t1.1350500000000001\nFS53962190.CDS.1\t1.12183\nZL767849.CDS.1\t1.07654\nQS900083.CDS.1\t1.05693\nND80117744.CDS.1\t1.04862\nKP22241601.CDS.2\t0.9468399999999999\nYM29518703.CDS.1\t0.9122\nBQ99602892.CDS.1\t0.7989900000000001\nBE618688.CDS.1\t0.78614\n'

# 03 - 

Change exercise 2 in the following way: 
    There is an input file, negative_list.txt, which is a list of genes which can NOT be part of the output. 
    They are banned from your analysis. As can be seen, the genes are identified by their swissprot id. 
    In order to translate from swissprot id to accession number so you can relate it to the scores.txt, 
    you must use the input file translation.txt, where the first item on the line is a accession number, second item is the corresponding swissprot id.

In [22]:
import pandas as pd

def main():
    filename_scores = "scores.txt"
    filename_negative = "negative_list.txt"
    filename_translation = "translation.txt"

    # Load data
    df_scores = pd.read_csv(
        filename_scores,
        sep="\t",
        names=["accession", "1", "2", "3", "4", "5", "6"],
        dtype={"accession": "string"},
    )

    df_negative = pd.read_csv(
        filename_negative,
        sep="\t",
        names=["swissprot"],
        dtype={"swissprot": "string"},
    )

    df_translation = pd.read_csv(
        filename_translation,
        sep="\t",
        names=["accession", "swissprot"],
        dtype={"accession": "string", "swissprot": "string"},
    )

    df_negative_map = df_negative.merge(df_translation, on="swissprot", how="left") # Get the accessions for the negative list
    negative_accessions = df_negative_map["accession"].dropna().unique() # Drop if no match
    df_filtered = df_scores[~df_scores["accession"].isin(negative_accessions)].copy() # Filter out the negative accessions

    # Sum the cols and order
    score_cols = ["1", "2", "3", "4", "5", "6"]
    df_filtered[score_cols] = df_filtered[score_cols].apply(
        pd.to_numeric, errors="coerce"
    ).fillna(0)
    df_filtered["Total"] = df_filtered[score_cols].sum(axis=1)
    df_ranked = df_filtered[["accession", "Total"]].sort_values("Total", ascending=False)

    # Save outputs
    df_filtered.to_csv("scores_filtered.txt", sep="\t", index=False)      # includes Total
    df_extreme = pd.concat([df_ranked.head(10), df_ranked.tail(10)])
    df_extreme.to_csv("scores_extreme.txt", sep="\t", index=False)

    print(df_extreme.to_string(index=False))

if __name__ == "__main__":
    main()

       accession   Total
  LF808489.CDS.1 5.27661
  KI133588.CDS.2 5.24153
JB80314549.CDS.3 5.17875
    J21782.CDS.3 5.15755
UY37981324.CDS.1 5.11245
CR97613151.CDS.1 5.09985
      TF58157495 5.07904
    N33521.CDS.1 5.03746
    E15922.CDS.2 5.02731
HD25632497.CDS.1 5.00469
  LL231571.CDS.2 1.14971
JP12357726.CDS.1 1.14921
  IE553374.CDS.1 1.13505
FS53962190.CDS.1 1.12183
  ZL767849.CDS.1 1.07654
  QS900083.CDS.1 1.05693
KP22241601.CDS.2 0.94684
YM29518703.CDS.1 0.91220
BQ99602892.CDS.1 0.79899
  BE618688.CDS.1 0.78614


# 04 - 

Change exercise 2 in the following way: Make the program work no matter how many numbers there are on every line. It must be the same number of numbers, i.e. in one file it could be 10 numbers on every line, in another file it could be 7 numbers per line.

In [25]:
import pandas as pd

def main():
    filename = "scores.txt"

    df = pd.read_csv(filename, sep="\t", header=None)
    df = df.rename(columns={0: "id"})

    df["Total"] = df.iloc[:, 1:].sum(axis=1)
    df = df[['id', 'Total']]
    df = df.sort_values(by="Total", ascending=False)

    df_top = df.head(10)
    df_tail = df.tail(10)
    df_extreme = pd.concat([df_top, df_tail])
    df_extreme.to_csv("scores_extreme.txt", sep="\t", index=False)

    print(df_extreme)

main()

                    id    Total
3643    LF808489.CDS.1  5.27661
0       KI133588.CDS.2  5.24153
2310  JB80314549.CDS.3  5.17875
4836      J21782.CDS.3  5.15755
416   UY37981324.CDS.1  5.11245
432   CR97613151.CDS.1  5.09985
3868        TF58157495  5.07904
2795      N33521.CDS.1  5.03746
2658      E15922.CDS.2  5.02731
2842  HD25632497.CDS.1  5.00469
3902  JP12357726.CDS.1  1.14921
2769    IE553374.CDS.1  1.13505
4796  FS53962190.CDS.1  1.12183
3313    ZL767849.CDS.1  1.07654
1136    QS900083.CDS.1  1.05693
2666  ND80117744.CDS.1  1.04862
3937  KP22241601.CDS.2  0.94684
1683  YM29518703.CDS.1  0.91220
3572  BQ99602892.CDS.1  0.79899
1601    BE618688.CDS.1  0.78614


# 05 - 

In [24]:
import pandas as pd

def main():
    filename = "scores.txt"

    # Load data
    df = pd.read_csv(filename, sep="\t", header=None)

    # Compute average and order
    df["Average"] = df.iloc[:, 1:].mean(axis=1)
    df = df[[0, "Average"]]
    df.columns = ["id", "Average"]
    df = df.sort_values(by="Average", ascending=False)

    # Save the top 10 in a new df
    df_top = df.head(10)
    df_tail = df.tail(10)
    df_extreme = pd.concat([df_top, df_tail])
    df_extreme.to_csv("scores_extreme.txt", sep="\t", index=False)

    print(df_extreme)

main()

       accession   Average
  LF808489.CDS.1  0.879435
  KI133588.CDS.2  0.873588
JB80314549.CDS.3  0.863125
    J21782.CDS.3  0.859592
UY37981324.CDS.1  0.852075
CR97613151.CDS.1  0.849975
      TF58157495  0.846507
    N33521.CDS.1  0.839577
    E15922.CDS.2  0.837885
HD25632497.CDS.1  0.834115
  LL231571.CDS.2  0.191618
JP12357726.CDS.1  0.191535
  IE553374.CDS.1  0.189175
FS53962190.CDS.1  0.186972
  ZL767849.CDS.1  0.179423
  QS900083.CDS.1  0.176155
KP22241601.CDS.2  0.157807
YM29518703.CDS.1  0.152033
BQ99602892.CDS.1  0.133165
  BE618688.CDS.1  0.131023


06 - 

Change exercise 2 in the following way: When you calculate the combined score the first number should weigh 50% more than the other numbers and the last should weigh 50% less.

In [27]:
import pandas as pd

def main():
    filename = "scores.txt"

    # Load data
    df = pd.read_csv(filename, sep="\t", names=['id', '1', '2', '3', '4', '5', '6'])

    # Apply weights and sum cols
    weights = [1.5, 1, 1, 1, 1, 0.5]
    df["Total"] = (df.iloc[:, 1:] * weights).sum(axis=1)

    # Order
    df = df[['id', 'Total']]
    df = df.sort_values(by="Total", ascending=False)

    # Save the top 10 in a new df
    df_top = df.head(10)
    df_tail = df.tail(10)
    df_extreme = pd.concat([df_top, df_tail])
    df_extreme.to_csv("scores_extreme.txt", sep="\t", index=False)

    print(df_extreme)

main()

                    id     Total
416   UY37981324.CDS.1  5.231955
0       KI133588.CDS.2  5.222055
3643    LF808489.CDS.1  5.219250
2310  JB80314549.CDS.3  5.176125
4836      J21782.CDS.3  5.146760
2658      E15922.CDS.2  5.132090
2842  HD25632497.CDS.1  5.096155
3748      F38888.CDS.1  5.029915
3868        TF58157495  4.999170
432   CR97613151.CDS.1  4.992815
3313    ZL767849.CDS.1  1.053470
1198    BH320550.CDS.2  1.044000
3937  KP22241601.CDS.2  0.930040
2769    IE553374.CDS.1  0.927600
337     LL231571.CDS.2  0.866565
1241        UG92888132  0.866510
3572  BQ99602892.CDS.1  0.845930
1136    QS900083.CDS.1  0.829260
2666  ND80117744.CDS.1  0.804800
1601    BE618688.CDS.1  0.736255


# 07

In [28]:
import pandas as pd

def main():
    filename = "scores.txt"

    # Load data
    df = pd.read_csv(filename, sep="\t", names=['id', '1', '2', '3', '4', '5', '6'])

    # Weighted sum cols (linear sliding scale)
    values = df.iloc[:, 1:]
    N = values.shape[1]  # number of numbers on the line

    # W = B - (B - E) * (P - 1) / (N - 1)
    B, E = 1.5, 0.5
    if N == 1:
        weights = pd.Series([B], index=values.columns)
    else:
        P = pd.Series(range(1, N + 1), index=values.columns)  # positions 1..N
        weights = B - (B - E) * (P - 1) / (N - 1)

    df["Total"] = values.mul(weights, axis=1).sum(axis=1)
    df = df[['id', 'Total']]
    df = df.sort_values(by="Total", ascending=False)

    # Save the top 10 and bottom 10 in a new df
    df_top = df.head(10)
    df_tail = df.tail(10)
    df_extreme = pd.concat([df_top, df_tail])
    df_extreme.to_csv("scores_extreme.txt", sep="\t", index=False)

    print(df_extreme)

main()

                    id     Total
3643    LF808489.CDS.1  5.294919
416   UY37981324.CDS.1  5.279049
2658      E15922.CDS.2  5.201149
0       KI133588.CDS.2  5.176149
2310  JB80314549.CDS.3  5.146801
4836      J21782.CDS.3  5.132851
3748      F38888.CDS.1  5.101997
2842  HD25632497.CDS.1  5.100983
3868        TF58157495  4.990356
4975    PA237885.CDS.1  4.987901
979   IW38005607.CDS.2  1.006034
1438  GB94053773.CDS.2  0.980373
2769    IE553374.CDS.1  0.939649
337     LL231571.CDS.2  0.898605
1241        UG92888132  0.850166
3572  BQ99602892.CDS.1  0.847787
1136    QS900083.CDS.1  0.835945
3937  KP22241601.CDS.2  0.822898
1601    BE618688.CDS.1  0.819844
2666  ND80117744.CDS.1  0.779526


08 - 

In [29]:
import pandas as pd

def main():
    filename = "scores.txt"

    # Load data
    df = pd.read_csv(filename, sep="\t", names=['id', '1', '2', '3', '4', '5', '6'])

    # Sum cols and order
    df["Total"] = df.iloc[:, 1:].sum(axis=1)
    df = df[['id', 'Total']]
    df = df.sort_values(by="Total", ascending=False)

    # Save the top 10 in a new df
    df_top = df.head(10)
    df_top.to_csv("scores_top10.txt", sep="\t", index=False)

    print(df_top)

main()

                    id    Total
3643    LF808489.CDS.1  5.27661
0       KI133588.CDS.2  5.24153
2310  JB80314549.CDS.3  5.17875
4836      J21782.CDS.3  5.15755
416   UY37981324.CDS.1  5.11245
432   CR97613151.CDS.1  5.09985
3868        TF58157495  5.07904
2795      N33521.CDS.1  5.03746
2658      E15922.CDS.2  5.02731
2842  HD25632497.CDS.1  5.00469


09 - 

In [30]:
########### ----------- Exercise 09 ----------- ###########
import pandas as pd

def main():
    filename = "scores.txt"
    k = 10
    cols = ['id', '1', '2', '3', '4', '5', '6']
    best = pd.DataFrame(columns=['id', 'Total'])

    for chunk in pd.read_csv(filename, sep="\t", names=cols, chunksize=200_000):
        # Sum cols and order
        chunk["Total"] = chunk.iloc[:, 1:].sum(axis=1)
        chunk = chunk[['id', 'Total']]

        best = pd.concat([best, chunk], ignore_index=True)
        best = best.nlargest(k, "Total")  # keep only top k so memory stays tiny

    # Save the top 10
    best = best.sort_values("Total", ascending=False)
    best.to_csv("scores_top10.txt", sep="\t", index=False)

    print(best)

main()

                    id    Total
3643    LF808489.CDS.1  5.27661
0       KI133588.CDS.2  5.24153
2310  JB80314549.CDS.3  5.17875
4836      J21782.CDS.3  5.15755
416   UY37981324.CDS.1  5.11245
432   CR97613151.CDS.1  5.09985
3868        TF58157495  5.07904
2795      N33521.CDS.1  5.03746
2658      E15922.CDS.2  5.02731
2842  HD25632497.CDS.1  5.00469


/tmp/ipykernel_36261/799936632.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  best = pd.concat([best, chunk], ignore_index=True)
